**(To be executed in kaggle)**

# **Approche Extractive**

## **Configuration**

### **Install Libraires**

In [3]:
# !pip install -q --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# !pip uninstall -y torch torchvision torchaudio transformers bert-score
# !pip install torch==2.2.2 transformers==4.38.2 bert-score==0.3.13

### **Check GPU availability**

In [4]:
# import torch
# print(torch.__version__)
# print(torch.cuda.is_available())

# print("GPU available:", torch.cuda.is_available())
# print("GPU count:", torch.cuda.device_count())

# for i in range(torch.cuda.device_count()):
#     print(i, torch.cuda.get_device_name(i))

## **Setup Pipelines**

### **1. Sentence Extraction**

In [5]:
import numpy as np
import networkx as nx
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import (
    CountVectorizer,
    TfidfVectorizer
)

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

# =========================================================
# NLTK Downloads
# =========================================================

nltk.download("punkt")
nltk.download("stopwords")

STOPWORDS = set(stopwords.words("english"))


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\exe\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\exe\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [6]:

# =========================================================
# STEP 1 — Sentence Extraction
# =========================================================

class SentenceExtractor(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        documents = []

        for text in X:

            sentences = sent_tokenize(text)

            documents.append({
                "original_text": text,
                "sentences": sentences
            })

        return documents

### **2. Tokenization + Cleaning**

In [7]:


# =========================================================
# STEP 2 — Tokenization + Cleaning
# =========================================================

class Tokenizer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        for doc in X:

            cleaned_sentences = []

            for sentence in doc["sentences"]:

                tokens = word_tokenize(sentence.lower())

                tokens = [
                    token
                    for token in tokens
                    if token.isalnum()
                    and token not in STOPWORDS
                ]

                cleaned_sentences.append(
                    " ".join(tokens)
                )

            doc["cleaned_sentences"] = cleaned_sentences

        return X



### **3. Vectorization**

In [8]:

# =========================================================
# STEP 3 — Vectorization
# =========================================================

class SentenceVectorizer(BaseEstimator, TransformerMixin):

    def __init__(self, vectorizer_type="tfidf"):

        self.vectorizer_type = vectorizer_type

        if vectorizer_type == "bow":
            self.vectorizer = CountVectorizer()

        elif vectorizer_type == "tfidf":
            self.vectorizer = TfidfVectorizer()

        else:
            raise ValueError(
                "vectorizer_type must be 'bow' or 'tfidf'"
            )

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        for doc in X:

            sentence_vectors = self.vectorizer.fit_transform(
                doc["cleaned_sentences"]
            )

            doc["sentence_vectors"] = sentence_vectors

        return X


### **4. Similarity Matrix**

In [9]:

# =========================================================
# STEP 4 — Similarity Matrix
# =========================================================

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

class SimilarityMatrix(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        for doc in X:

            sim_matrix = cosine_similarity(
                doc["sentence_vectors"]
            )

            np.fill_diagonal(sim_matrix, 0)

            doc["similarity_matrix"] = sim_matrix

        return X



### **5. TextRank Summarizer**

In [10]:


# =========================================================
# STEP 5 — TextRank Summarizer
# =========================================================

class TextRankSummarizer(BaseEstimator, TransformerMixin):

    def __init__(self, top_k=3):

        self.top_k = top_k

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        summaries = []

        for doc in X:

            sim_matrix = doc["similarity_matrix"]

            graph = nx.from_numpy_array(sim_matrix)

            scores = nx.pagerank(
                graph,
                alpha=0.85
            )

            ranked_sentences = sorted(
                (
                    (scores[i], sentence, i)
                    for i, sentence in enumerate(doc["sentences"])
                ),
                reverse=True
            )

            top_sentences = sorted(
                ranked_sentences[:self.top_k],
                key=lambda x: x[2]
            )

            summary = " ".join(
                sentence
                for _, sentence, _
                in top_sentences
            )

            summaries.append(summary)

        return summaries


### **6. Create Pipelines**

In [17]:
import pandas as pd

df  = pd.read_csv("../../Datasets/MultiClinSum/multiclinsum_dataset.csv").sample(50)

In [18]:
def build_pipeline(vectorizer_type="tfidf", top_k=3):

    pipeline = Pipeline([
        ("sentence_extraction", SentenceExtractor()),
        ("tokenization", Tokenizer()),
        ("vectorization", SentenceVectorizer(vectorizer_type)),
        ("similarity_matrix", SimilarityMatrix()),
        ("textrank", TextRankSummarizer(top_k=top_k))
    ])
    
    pipeline.fit(df["content"], df["summary"])

    import joblib
    joblib.dump(pipeline, vectorizer_type + "_summarizer.pkl")

    return pipeline

In [19]:
# =========================================================
# CREATE MULTIPLE PIPELINES
# =========================================================

pipelines = {

    "BoW": build_pipeline(
        vectorizer_type="bow",
        top_k=3
    ),

    "TF-IDF": build_pipeline(
        vectorizer_type="tfidf",
        top_k=3
    )
}

## **Load Data**

In [12]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/nizarmadih/multiclinsum/multiclinsum_dataset.csv", index_col=False)
df = df.drop(columns="Unnamed: 0")

documents = df["content"].tolist()

In [13]:
df

,complete_id,id,unique_id,language,quality,source,content,summary
0,1_en_gs,1,0,en,gs,multiclinsum,A 52-year-old female patient from Wolkiet (Nor...,A 52-year-old known female patient with a toxi...
1,1_en_ls,1,1,en,ls,multiclinsum,A healthy 53-year-old Japanese man experienced...,A 53-year-old Japanese man experienced sudden ...
2,2_en_gs,2,2,en,gs,multiclinsum,"35-year-old man, occupation: sandblaster for e...","We present the case of a 35-year-old man, a sa..."
3,2_en_ls,2,3,en,ls,multiclinsum,Written informed consent was obtained from the...,An ambulatory open inguinal herniorrhaphy was ...
4,3_en_gs,3,4,en,gs,multiclinsum,We present a case of a 59-year-old lady with a...,A 59-year-old woman was referred to the ophtha...
...,...,...,...,...,...,...,...,...
52983,25898_fr_ls,25898,52983,fr,ls,multiclinsum,Un homme de 48 ans atteint d'ESKD a présenté u...,Nous décrivons un cas de neuropathie périphéri...
52984,25899_fr_ls,25899,52984,fr,ls,multiclinsum,"Une femme caucasienne de 37 ans, auparavant en...",Une femme de 37 ans en bonne santé a présenté ...
52985,25900_fr_ls,25900,52985,fr,ls,multiclinsum,Une Chinoise de 55 ans se présente avec une do...,Une Chinoise de 55 ans s'est présentée avec de...
52986,25901_fr_ls,25901,52986,fr,ls,multiclinsum,La patiente était une fille de 3 mois née à 40...,La patiente dans ce cas était une fille de 3 m...


## **Setup Metric Calculation**

### **ROUGE**

In [14]:
!pip install rouge_score bert_score

In [15]:
# =========================================================
# IMPORTS
# =========================================================
from rouge_score import rouge_scorer
from bert_score import score as bertscore

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer


In [16]:


# =========================================================
# ROUGE METRICS
# =========================================================

def compute_rouge(predictions, references):

    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        use_stemmer=True
    )

    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []

    for pred, ref in zip(predictions, references):

        scores = scorer.score(ref, pred)

        rouge1_scores.append(
            scores["rouge1"].fmeasure
        )

        rouge2_scores.append(
            scores["rouge2"].fmeasure
        )

        rougeL_scores.append(
            scores["rougeL"].fmeasure
        )

    return {

        "ROUGE-1": np.mean(rouge1_scores),

        "ROUGE-2": np.mean(rouge2_scores),

        "ROUGE-L": np.mean(rougeL_scores)
    }



### **BERTScore**

In [17]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [18]:

# =========================================================
# BERTScore
# =========================================================

from bert_score import score as bertscore
import torch

def compute_bertscore(predictions, references):

    device = "cuda" if torch.cuda.is_available() else "cpu"

    P, R, F1 = bertscore(
        predictions,
        references,
        lang="en",
        device=device,
        batch_size=8,
        model_type="distilbert-base-uncased",  # MUCH lighter
        verbose=True
    )

    return {
        "BERTScore-P": P.mean().item(),
        "BERTScore-R": R.mean().item(),
        "BERTScore-F1": F1.mean().item()
    }




def compute_bertscore_by_lang(preds, refs, langs):

    from bert_score import score as bertscore

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # split by language
    en_preds, en_refs = [], []
    fr_preds, fr_refs = [], []

    for p, r, l in zip(preds, refs, langs):
        if l == "fr":
            fr_preds.append(p)
            fr_refs.append(r)
        else:
            en_preds.append(p)
            en_refs.append(r)

    results = {}

    # ---------------- ENGLISH ----------------
    if len(en_preds) > 0:
        P, R, F1 = bertscore(
            en_preds,
            en_refs,
            model_type="distilbert-base-uncased",
            device=device,
            batch_size=8,
            verbose=True
        )
        results["BERTScore-F1-en"] = F1.mean().item()

    # ---------------- FRENCH ----------------
    if len(fr_preds) > 0:
        P, R, F1 = bertscore(
            fr_preds,
            fr_refs,
            lang="fr",
            device=device,
            batch_size=8,
            verbose=True
        )
        results["BERTScore-F1-fr"] = F1.mean().item()

    # ---------------- GLOBAL SCORE ----------------
    if len(en_preds) + len(fr_preds) > 0:
        results["BERTScore-F1"] = (
            results.get("BERTScore-F1-en", 0) * len(en_preds) +
            results.get("BERTScore-F1-fr", 0) * len(fr_preds)
        ) / (len(en_preds) + len(fr_preds))

    return results

import torch
from bert_score import score as bertscore

def compute_bertscore_chunk(preds, refs, langs, batch_size=4):

    device = "cuda" if torch.cuda.is_available() else "cpu"

    en_p, en_r = [], []
    fr_p, fr_r = [], []

    for p, r, l in zip(preds, refs, langs):
        if l == "fr":
            fr_p.append(p); fr_r.append(r)
        else:
            en_p.append(p); en_r.append(r)

    results = {}

    if en_p:
        _, _, F1 = bertscore(
            en_p, en_r,
            model_type="distilbert-base-uncased",
            device=device,
            batch_size=batch_size,
            verbose=True
        )
        results["en"] = F1.mean().item()

    if fr_p:
        _, _, F1 = bertscore(
            fr_p, fr_r,
            model_type="bert-base-multilingual-cased",
            device=device,
            batch_size=batch_size,
            verbose=True
        )
        results["fr"] = F1.mean().item()

    torch.cuda.empty_cache()
    return results

### **Compression Ration**

In [19]:

# =========================================================
# COMPRESSION RATIO
# =========================================================
# Measures summary compactness
#
# Lower ratio = stronger compression
# =========================================================

def compute_compression_ratio(
    original_texts,
    summaries
):

    ratios = []

    for original, summary in zip(
        original_texts,
        summaries
    ):

        original_words = len(original.split())
        summary_words = len(summary.split())

        if summary_words == 0:
            ratios.append(0)
        else:
            ratios.append(
                original_words / summary_words
            )

    return {
        "Compression-Ratio": np.mean(ratios)
    }


### **Redundancy Metric**

In [20]:


# =========================================================
# REDUNDANCY METRIC
# =========================================================
# Measures sentence repetition inside summaries
#
# Lower = better
# =========================================================

def compute_redundancy(summaries):

    redundancy_scores = []

    for summary in summaries:

        sentences = [
            s.strip()
            for s in summary.split(".")
            if s.strip()
        ]

        if len(sentences) <= 1:
            redundancy_scores.append(0)
            continue

        vectorizer = TfidfVectorizer()

        vectors = vectorizer.fit_transform(
            sentences
        )

        sim_matrix = cosine_similarity(vectors)

        upper_triangle = sim_matrix[
            np.triu_indices_from(
                sim_matrix,
                k=1
            )
        ]

        redundancy_scores.append(
            upper_triangle.mean()
        )

    return {
        "Redundancy": np.mean(
            redundancy_scores
        )
    }



### **Extractive Coverage**

In [21]:

# =========================================================
# EXTRACTIVE COVERAGE METRIC
# =========================================================
# Useful for extractive summarization
#
# Measures how much summary content
# exists in original text
#
# Higher = better factual grounding
# =========================================================

def compute_extractive_coverage(
    original_texts,
    summaries
):

    coverage_scores = []

    for original, summary in zip(
        original_texts,
        summaries
    ):

        original_words = set(
            original.lower().split()
        )

        summary_words = summary.lower().split()

        if len(summary_words) == 0:
            coverage_scores.append(0)
            continue

        overlap = sum(
            word in original_words
            for word in summary_words
        )

        coverage_scores.append(
            overlap / len(summary_words)
        )

    return {
        "Extractive-Coverage": np.mean(
            coverage_scores
        )
    }


### **All Metrics**

In [22]:

# =========================================================
# GLOBAL EVALUATION FUNCTION
# =========================================================

def evaluate_summaries(
    original_texts,
    predicted_summaries,
    reference_summaries
):

    results = {}

    # -------------------------
    # ROUGE
    # -------------------------

    results.update(
        compute_rouge(
            predicted_summaries,
            reference_summaries
        )
    )

    # -------------------------
    # BERTScore
    # -------------------------

    results.update(
        compute_bertscore(
            predicted_summaries,
            reference_summaries
        )
    )

    # -------------------------
    # Compression Ratio
    # -------------------------

    results.update(
        compute_compression_ratio(
            original_texts,
            predicted_summaries
        )
    )

    # -------------------------
    # Redundancy
    # -------------------------

    results.update(
        compute_redundancy(
            predicted_summaries
        )
    )

    # -------------------------
    # Extractive Coverage
    # -------------------------

    results.update(
        compute_extractive_coverage(
            original_texts,
            predicted_summaries
        )
    )


    return pd.DataFrame([results])


In [25]:
import json
import time

def generate_and_save(pipeline, documents, references, languages, out_path):

    start = time.time()

    with open(out_path, "w") as f:

        for i, doc in enumerate(documents):

            summary = pipeline.fit_transform([doc])[0]

            record = {
                "id": i,
                "pred": summary,
                "ref": references[i],
                "lang": languages[i]
            }

            f.write(json.dumps(record) + "\n")

            if (i + 1) % 100 == 0:
                print(f"[GEN] {i+1}/{len(documents)}")

    print(f"Generation done in {time.time() - start:.2f}s")

def stream_jsonl(path):
    with open(path, "r") as f:
        for line in f:
            yield json.loads(line)

import gc

def evaluate_from_disk(path, chunk_size=128):

    results_all = []

    preds, refs, langs = [], [], []

    for item in stream_jsonl(path):

        preds.append(item["pred"])
        refs.append(item["ref"])
        langs.append(item["lang"])

        if len(preds) == chunk_size:

            res = compute_bertscore_chunk(preds, refs, langs)
            results_all.append(res)

            preds, refs, langs = [], [], []

            gc.collect()
            torch.cuda.empty_cache()

            print("[EVAL] chunk done")

    # last chunk
    if preds:
        results_all.append(compute_bertscore_chunk(preds, refs, langs))

    return results_all

from rouge_score import rouge_scorer

def compute_rouge_stream(path, chunk_size=256):

    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

    r1, r2, rl = [], [], []

    preds, refs = [], []

    for item in stream_jsonl(path):

        preds.append(item["pred"])
        refs.append(item["ref"])

        if len(preds) == chunk_size:

            for p, r in zip(preds, refs):
                score = scorer.score(r, p)
                r1.append(score["rouge1"].fmeasure)
                r2.append(score["rouge2"].fmeasure)
                rl.append(score["rougeL"].fmeasure)

            preds, refs = [], []

    return {
        "ROUGE-1": sum(r1)/len(r1),
        "ROUGE-2": sum(r2)/len(r2),
        "ROUGE-L": sum(rl)/len(rl),
    }

## **Extractive Summarization + Metric Calculation**

In [28]:
import time
import pandas as pd
import gc
import torch
import json

all_results = []

languages = df["language"].tolist()
documents = df["content"].tolist()
reference_summaries = df["summary"].tolist()

all_predictions = json.load(
    open("/kaggle/input/datasets/nizarmadih/pred-results/predicted_summaries.json")
)

# =========================================================
# SAFE BERTSCORE WRAPPER
# =========================================================
def safe_bertscore(preds, refs, langs, chunk_size=128):

    from bert_score import score as bertscore

    results_en = []
    results_fr = []

    for i in range(0, len(preds), chunk_size):

        p_chunk = preds[i:i+chunk_size]
        r_chunk = refs[i:i+chunk_size]
        l_chunk = langs[i:i+chunk_size]

        en_p, en_r = [], []
        fr_p, fr_r = [], []

        for p, r, l in zip(p_chunk, r_chunk, l_chunk):
            if l == "fr":
                fr_p.append(p)
                fr_r.append(r)
            else:
                en_p.append(p)
                en_r.append(r)

        device = "cuda" if torch.cuda.is_available() else "cpu"

        if en_p:
            _, _, f1 = bertscore(
                en_p, en_r,
                model_type="distilbert-base-uncased",
                device=device,
                batch_size=4,
                verbose=True
            )
            results_en.append(f1.mean().item())

        if fr_p:
            _, _, f1 = bertscore(
                fr_p, fr_r,
                model_type="bert-base-multilingual-cased",
                device=device,
                batch_size=4,
                verbose=True
            )
            results_fr.append(f1.mean().item())

        torch.cuda.empty_cache()
        gc.collect()

    all_scores = results_en + results_fr

    return {
        "BERTScore-F1": sum(all_scores) / max(1, len(all_scores))
    }


# =========================================================
# MAIN LOOP
# =========================================================

for name, predicted_summaries in all_predictions.items():

    print("\n" + "=" * 80)
    print(f"[EVALUATION] PIPELINE: {name}")
    print("=" * 80)

    eval_start = time.time()

    # ---------------- BERTScore ----------------
    print("[METRICS] BERTScore (EN + FR)...")
    t0 = time.time()

    bert_results = safe_bertscore(
        predicted_summaries,
        reference_summaries,
        languages
    )

    print(f"BERTScore done in {time.time() - t0:.2f}s")

    # ---------------- ROUGE ----------------
    print("[METRICS] ROUGE...")
    t0 = time.time()

    rouge_results = compute_rouge(
        predicted_summaries,
        reference_summaries
    )

    print(f"ROUGE done in {time.time() - t0:.2f}s")

    # ---------------- COMPRESSION ----------------
    print("[METRICS] Compression...")
    t0 = time.time()

    compression_results = compute_compression_ratio(
        documents,
        predicted_summaries
    )

    print(f"Compression done in {time.time() - t0:.2f}s")

    # ---------------- MERGE ----------------
    results = {}
    results.update(rouge_results)
    results.update(bert_results)
    results.update(compression_results)

    results["Pipeline"] = name

    # FIXED LINE (no crash)
    results["Generation-Time"] = None

    results["Evaluation-Time"] = time.time() - eval_start

    all_results.append(results)

    gc.collect()
    torch.cuda.empty_cache()


# =========================================================
# FINAL TABLE
# =========================================================

final_results = pd.DataFrame(all_results)

print(final_results)

sorted_results = final_results.sort_values(
    by="BERTScore-F1",
    ascending=False
)

print(sorted_results)

final_results.to_csv("extractive_results.csv", index=False)


[EVALUATION] PIPELINE: BoW
[METRICS] BERTScore (EN + FR)...
calculating scores...
computing bert embedding.


  0%|          | 0/64 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/32 [00:00<?, ?it/s]

done in 1.63 seconds, 78.48 sentences/sec
calculating scores...
computing bert embedding.


...

done in 3.60 seconds, 35.53 sentences/sec
calculating scores...
computing bert embedding.


  0%|          | 0/62 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/31 [00:00<?, ?it/s]

done in 3.37 seconds, 36.74 sentences/sec
BERTScore done in 1537.17s
[METRICS] ROUGE...
ROUGE done in 287.31s
[METRICS] Compression...
Compression done in 2.77s
    ROUGE-1   ROUGE-2   ROUGE-L  BERTScore-F1  Compression-Ratio Pipeline  \
0  0.340628  0.124755  0.209753      0.758018           7.658707      BoW   
1  0.342247  0.126204  0.209753      0.757958           7.645997   TF-IDF   

  Generation-Time  Evaluation-Time  
0            None      1793.401716  
1            None      1827.241289  
    ROUGE-1   ROUGE-2   ROUGE-L  BERTScore-F1  Compression-Ratio Pipeline  \
0  0.340628  0.124755  0.209753      0.758018           7.658707      BoW   
1  0.342247  0.126204  0.209753      0.757958           7.645997   TF-IDF   

  Generation-Time  Evaluation-Time  
0            None      1793.401716  
1            None      1827.241289  


## **Export Pipelines**

In [34]:
import joblib

joblib.dump(
    pipelines["BoW"],
    "BoW_summarizer.pkl"
)

['BoW_summarizer.pkl']

In [ ]:
import joblib

joblib.dump(
    pipelines["TF-IDF"],
    "tfidf_summarizer.pkl"
)

In [33]:
pipelines

{'BoW': Pipeline(steps=[('sentence_extraction', SentenceExtractor()),
                 ('tokenization', Tokenizer()),
                 ('vectorization', SentenceVectorizer(vectorizer_type='bow')),
                 ('similarity_matrix', SimilarityMatrix()),
                 ('textrank', TextRankSummarizer())]),
 'TF-IDF': Pipeline(steps=[('sentence_extraction', SentenceExtractor()),
                 ('tokenization', Tokenizer()),
                 ('vectorization', SentenceVectorizer()),
                 ('similarity_matrix', SimilarityMatrix()),
                 ('textrank', TextRankSummarizer())])}